# Chapter 11: Multi-Modal Perception Agents

**Book:** *30 Agents Every AI Engineer Must Build*
**Author:** Imran Ahmad
**Publisher:** Packt Publishing

---

This notebook implements the three multi-modal perception domains covered in Chapter 11:

1. **Vision-Language Agents** — Joint image-text reasoning using a visual encoder, alignment mechanism, and LLM
2. **Audio Processing Agents** — Speech transcription with mode-aware normalization and voice sentiment analysis via the VAD model
3. **Physical World Sensing Agents** — Sensor fusion, pattern-based event detection, and proportional control with deadband hysteresis

All three domains follow the **Sense-Model-Plan-Act** loop. The notebook auto-detects your environment and runs in **Simulation Mode** if no GPU or Hugging Face token is available.

> *Ref: Chapter 11, Multi-Modal Perception Agents — all sections*

In [ ]:
# ============================================================
# §0 — Environment Detection
# Ref: Technical Requirements (p.2)
# Author: Imran Ahmad
# ============================================================
# Detects GPU availability and Hugging Face token to determine
# whether to run in Simulation Mode (mock backends) or Live Mode
# (real model inference with transformers + torch).
# ============================================================

import os
import sys

# --- Load .env if present (cascading fallback) ---
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # python-dotenv not installed; fall through to env vars

# --- Detect environment capabilities ---
SIMULATION_REASONS = []

# Check 1: CUDA GPU availability
try:
    import torch
    HAS_CUDA = torch.cuda.is_available()
    if not HAS_CUDA:
        SIMULATION_REASONS.append("No CUDA GPU detected")
except ImportError:
    HAS_CUDA = False
    SIMULATION_REASONS.append("torch not installed")

# Check 2: Hugging Face token
HF_TOKEN = os.environ.get("HUGGINGFACE_TOKEN", "").strip()
if not HF_TOKEN or HF_TOKEN == "your_hugging_face_token_here":
    HAS_TOKEN = False
    SIMULATION_REASONS.append("No valid HUGGINGFACE_TOKEN in environment")
else:
    HAS_TOKEN = True

# --- Final mode decision ---
SIMULATION_MODE = not (HAS_CUDA and HAS_TOKEN)


In [ ]:
# ============================================================
# §0 — Mode Banner
# Ref: Technical Requirements (p.2)
# Author: Imran Ahmad
# ============================================================

from agent_logger import AgentLogger

if SIMULATION_MODE:
    reasons = "; ".join(SIMULATION_REASONS) if SIMULATION_REASONS else "Forced by configuration"
    AgentLogger.info(f"Simulation Mode active. Reasons: {reasons}")
    AgentLogger.info(
        "All agents will use mock backends from mock_backends.py. "
        "No GPU or API token required."
    )
else:
    AgentLogger.success("Live Mode active. CUDA GPU and Hugging Face token detected.")
    AgentLogger.info("Agents will use real model inference via transformers + torch.")

AgentLogger.info(f"Python {sys.version.split()[0]} | CUDA: {HAS_CUDA} | HF Token: {HAS_TOKEN}")


---

# Part 1: Vision-Language Agents

> *Ref: Architecture of Vision-Language Agents (p.3-4), Building a Vision Question-Answering Agent (p.5-7)*

Vision-Language agents pair a **visual encoder** (typically a Vision Transformer / ViT) with a **large language model** through an **alignment mechanism** that projects visual embeddings into the language model's token space. This three-component architecture enables the agent to reason jointly over image content and natural language inputs.

The key architectural components:

- **Visual Encoder (ViT):** Segments images into fixed-size patches (14×14 or 16×16 pixels), projects each into an embedding space, and captures local features and global context through self-attention.
- **Alignment Mechanism:** Projects visual embeddings into the LLM's token space via learned linear projections or MLPs, ensuring visual representations become "legible" to the reasoning engine.
- **Large Language Model:** Receives text prompts alongside aligned visual tokens, using cross-modal attention to ground reasoning in visual evidence.

We implement a `VisionQuestionAnsweringAgent` using **Chain-of-Thought (CoT)** prompting to force step-by-step visual analysis before committing to an answer.

In [ ]:
# ============================================================
# §1.1 — Generate Test Image (Programmatic)
# Ref: Building a Vision Question-Answering Agent (p.5)
# Author: Imran Ahmad
# ============================================================
# Creates a synthetic workspace image for the Vision agent demos.
# No external downloads required.
# ============================================================

import numpy as np
from PIL import Image, ImageDraw, ImageFont
import os

def create_test_workspace_image(path: str = "assets/sample_workspace.png") -> Image.Image:
    """
    Generate a synthetic workspace image with identifiable objects
    for the VisionQuestionAnsweringAgent demos.

    Objects drawn: desk surface, laptop, coffee cup, papers, desk lamp,
    two stylized person silhouettes (one partially occluded).

    Ref: Building a Vision Question-Answering Agent (p.5-6)
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    img = Image.new("RGB", (640, 480), color=(245, 240, 230))  # Warm beige background
    draw = ImageDraw.Draw(img)

    # Desk surface (brown rectangle)
    draw.rectangle([50, 250, 590, 460], fill=(139, 90, 43), outline=(100, 65, 30), width=2)

    # Laptop (gray rectangle with blue screen)
    draw.rectangle([220, 180, 420, 320], fill=(80, 80, 85), outline=(60, 60, 65), width=2)
    draw.rectangle([235, 190, 405, 280], fill=(30, 100, 180))  # Screen

    # Coffee cup (right side, on papers — "precariously balanced")
    # Papers first
    draw.rectangle([440, 260, 530, 310], fill=(255, 255, 255), outline=(180, 180, 180))
    draw.rectangle([445, 265, 535, 315], fill=(255, 255, 250), outline=(180, 180, 180))
    # Cup
    draw.ellipse([460, 240, 510, 270], fill=(200, 50, 50), outline=(150, 30, 30))
    draw.rectangle([465, 255, 505, 290], fill=(200, 50, 50), outline=(150, 30, 30))

    # Desk lamp (upper left)
    draw.rectangle([80, 160, 95, 260], fill=(60, 60, 60))  # Pole
    draw.polygon([(60, 140), (115, 140), (95, 170), (80, 170)], fill=(255, 220, 50))  # Shade

    # Person 1 (seated, center-left)
    draw.ellipse([160, 100, 200, 140], fill=(210, 180, 140))  # Head
    draw.rectangle([165, 140, 195, 200], fill=(50, 100, 150))  # Torso

    # Person 2 (partially occluded by bookshelf, right background)
    # Bookshelf
    draw.rectangle([520, 60, 600, 250], fill=(120, 70, 40), outline=(90, 50, 30), width=2)
    draw.rectangle([525, 70, 595, 110], fill=(100, 60, 35))  # Shelf
    draw.rectangle([525, 120, 595, 160], fill=(100, 60, 35))  # Shelf
    # Person behind bookshelf (partially visible)
    draw.ellipse([500, 80, 530, 110], fill=(190, 160, 130))  # Head
    draw.rectangle([505, 110, 525, 160], fill=(80, 130, 80))  # Torso (partially hidden)

    # Window light indicator (left side, yellowish glow)
    draw.rectangle([0, 0, 50, 480], fill=(255, 250, 220))

    # Label for educational clarity
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 12)
    except (OSError, IOError):
        font = ImageFont.load_default()
    draw.text((55, 465), "Synthetic test image — Chapter 11", fill=(120, 120, 120), font=font)

    img.save(path)
    AgentLogger.success(f"Test image saved to {path} ({img.size[0]}x{img.size[1]})")
    return img

test_image = create_test_workspace_image()
test_image


In [ ]:
# ============================================================
# §1.2 — VisionQuestionAnsweringAgent
# Ref: Building a Vision Question-Answering Agent (p.5-7)
# Author: Imran Ahmad
# ============================================================
# Implements the VQA agent with Chain-of-Thought prompting.
# Uses MockVLM + MockProcessor in Simulation Mode, or the real
# LLaVA 1.5 pipeline in Live Mode.
#
# Architecture (Ref: Architecture of Vision-Language Agents, p.3-4):
#   Image Input → Visual Encoder (ViT) → Alignment Mechanism →
#   Large Language Model (with cross-modal attention) → Output
# ============================================================

from agent_logger import AgentLogger, graceful_fallback

class VisionQuestionAnsweringAgent:
    """
    A vision-language agent capable of answering questions about images.
    Implements Chain-of-Thought reasoning for improved accuracy.

    In Simulation Mode, uses MockProcessor and MockVLM from
    mock_backends.py. In Live Mode, uses the real LLaVA 1.5
    pipeline from Hugging Face transformers.

    Ref: Building a Vision Question-Answering Agent (p.5-7)
    """

    def __init__(self, model_id: str = "llava-hf/llava-1.5-7b-hf"):
        """
        Initialize the agent with a pre-trained Vision-Language Model.

        Args:
            model_id: HuggingFace model identifier. LLaVA 1.5 provides
                      a good balance of capability and resource requirements.

        Ref: Initialization and Model Loading (p.5-6)
        """
        AgentLogger.info(f"Initializing Vision-Language Agent (model: {model_id})...")

        if SIMULATION_MODE:
            from mock_backends import MockProcessor, MockVLM
            self.processor = MockProcessor.from_pretrained(model_id)
            self.model = MockVLM.from_pretrained(model_id)
            AgentLogger.success("Agent ready (Simulation Mode)")
        else:
            import torch
            from transformers import AutoProcessor, LlavaForConditionalGeneration

            self.processor = AutoProcessor.from_pretrained(model_id)
            # Load with mixed precision (float16) for efficiency.
            # device_map="auto" distributes across available GPUs and
            # falls back to CPU when necessary.
            # Ref: Initialization and Model Loading (p.5-6)
            self.model = LlavaForConditionalGeneration.from_pretrained(
                model_id,
                torch_dtype=torch.float16,
                low_cpu_mem_usage=True,
                device_map="auto",
            )
            AgentLogger.success("Agent ready (Live Mode)")

        self.conversation_history = []

    @graceful_fallback(max_retries=2, base_delay=0.5)
    def answer_question(
        self,
        image: "Image.Image",
        question: str,
        use_chain_of_thought: bool = True,
    ) -> dict:
        """
        Answer a question about the provided image.

        Args:
            image: PIL Image object to analyze.
            question: Natural language question about the image.
            use_chain_of_thought: Whether to use step-by-step reasoning.

        Returns:
            Dictionary containing 'answer' and optionally 'reasoning'.

        Ref: Building a Vision Question-Answering Agent (p.5-7)
        """
        AgentLogger.info(f"answer_question called: '{question[:60]}...'")

        # Validate input
        if image is None:
            raise TypeError("NoneType image received — cannot process a null image input")

        # Construct prompt based on reasoning strategy
        if use_chain_of_thought:
            cot_instruction = self._build_cot_prompt(question)
            prompt = f"USER: <image>\n{cot_instruction}\nASSISTANT:"
        else:
            prompt = f"USER: <image>\n{question}\nASSISTANT:"

        # Process inputs: tokenize text and preprocess image
        inputs = self.processor(
            text=prompt,
            images=image,
            return_tensors="pt",
        ).to(self.model.device)

        # Generate response with deterministic decoding
        if SIMULATION_MODE:
            outputs = self.model.generate(**inputs, max_new_tokens=512)
        else:
            import torch
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=512,
                    do_sample=False,  # Greedy decoding for reproducibility
                )

        # Decode and parse
        full_response = self.processor.decode(outputs[0], skip_special_tokens=True)
        response_text = full_response.split("ASSISTANT:")[-1].strip()
        result = self._parse_response(response_text, use_chain_of_thought)

        AgentLogger.success(
            f"answer_question completed. "
            f"{'CoT reasoning extracted.' if 'reasoning' in result else 'Direct answer.'}"
        )
        return result

    def _build_cot_prompt(self, question: str) -> str:
        """
        Construct a Chain-of-Thought prompt for visual reasoning.

        The structured format guides the model through systematic
        analysis rather than pattern-matching to likely answers.

        Ref: Chain-of-Thought prompting pattern (p.10-12)
        """
        return (
            f'Analyze the image carefully and answer the following question: '
            f'"{question}"\n\n'
            f'Please think step by step:\n'
            f'1. First, identify the relevant objects or features visible in the image.\n'
            f'2. Then, examine the specific details needed to answer the question.\n'
            f'3. Finally, provide your answer based on the visual evidence.\n\n'
            f'Format your response as:\n'
            f'Reasoning: [Your step-by-step analysis]\n'
            f'Therefore, the answer is: [Your final answer]'
        )

    def _parse_response(self, response: str, has_reasoning: bool) -> dict:
        """
        Extract structured output from model response.

        When CoT is enabled, extracts the reasoning trace separately
        from the final answer. This separation enables explainability
        and supports debugging.

        Ref: Parsing and Structured Output (p.12-13)
        """
        if has_reasoning and "Therefore, the answer is:" in response:
            parts = response.split("Therefore, the answer is:")
            return {
                "reasoning": parts[0].replace("Reasoning:", "").strip(),
                "answer": parts[1].strip().rstrip("."),
            }
        return {"answer": response.strip()}

AgentLogger.success("VisionQuestionAnsweringAgent class defined.")


In [ ]:
# ============================================================
# §1.3 — Initialize the Vision-Language Agent
# Ref: Building a Vision Question-Answering Agent (p.5-6)
# Author: Imran Ahmad
# ============================================================

vqa_agent = VisionQuestionAnsweringAgent()


In [ ]:
# ============================================================
# §1.4 — Demo: Describe Workspace
# Ref: Building a Vision Question-Answering Agent (p.5-7)
# Author: Imran Ahmad
# ============================================================
# Scenario: Ask the agent to describe the synthetic workspace image.
# Tests the CoT reasoning pipeline end-to-end.
# ============================================================

result_describe = vqa_agent.answer_question(
    image=test_image,
    question="Describe this workspace in detail.",
    use_chain_of_thought=True,
)

print("\n--- Vision Agent: Describe Workspace ---")
if "reasoning" in result_describe:
    print(f"Reasoning: {result_describe['reasoning'][:200]}...")
print(f"Answer: {result_describe['answer']}")


In [ ]:
# ============================================================
# §1.5 — Demo: Count People
# Ref: Building a Vision Question-Answering Agent (p.5-7)
# Author: Imran Ahmad
# ============================================================
# Scenario: Count partially occluded objects — a task that
# demonstrates why direct visual access matters versus textual
# descriptions (Ref: Architecture of VL Agents, p.4).
# ============================================================

result_count = vqa_agent.answer_question(
    image=test_image,
    question="How many people are visible in this image? Count carefully.",
    use_chain_of_thought=True,
)

print("\n--- Vision Agent: Count People ---")
if "reasoning" in result_count:
    print(f"Reasoning: {result_count['reasoning'][:200]}...")
print(f"Answer: {result_count['answer']}")


In [ ]:
# ============================================================
# §1.6 — Demo: Spatial Relationships
# Ref: Integration Patterns and Production Considerations (p.6-7)
# Author: Imran Ahmad
# ============================================================
# Scenario: Analyze spatial layout — demonstrates cross-modal
# attention grounding linguistic reasoning in visual evidence.
# ============================================================

result_spatial = vqa_agent.answer_question(
    image=test_image,
    question="Describe the spatial relationship between the laptop, coffee cup, and desk lamp.",
    use_chain_of_thought=True,
)

print("\n--- Vision Agent: Spatial Relationships ---")
if "reasoning" in result_spatial:
    print(f"Reasoning: {result_spatial['reasoning'][:200]}...")
print(f"Answer: {result_spatial['answer']}")


### Integration Patterns and Production Considerations

> *Ref: Integration Patterns and Production Considerations (p.6-7)*

The `VisionQuestionAnsweringAgent` above uses a **direct integration** pattern where the model is loaded locally. In production, three architectural patterns are common:

- **Adapter-based integration:** Lightweight projection layers between frozen encoder and frozen LLM. Less than 1% of parameters trained. Efficient but limited on out-of-distribution visual concepts.
- **Cross-attention integration:** Dedicated attention layers for language tokens to attend to visual encoder outputs. Models like Flamingo demonstrate strong few-shot visual learning.
- **Early fusion:** Visual and textual tokens concatenated into a single sequence. Maximum cross-modal reasoning potential but increased computational overhead.

**Latency management techniques:** image resolution scaling (1024→336 cuts tokens by ~90%), patch pruning, speculative decoding, and visual embedding caching.

In [ ]:
# ============================================================
# §1.7 — Error Demo: Graceful Failure on Invalid Input
# Ref: Building a Vision Question-Answering Agent (p.5-7)
# Author: Imran Ahmad
# ============================================================
# Demonstrates the @graceful_fallback decorator in action.
# Passing None as the image triggers a TypeError, which the
# decorator catches, logs in RED, retries, and returns a
# structured fallback dictionary.
# ============================================================

AgentLogger.info("Demonstrating error handling with None image input...")

result_error = vqa_agent.answer_question(
    image=None,
    question="Describe this image.",
    use_chain_of_thought=True,
)

print("\n--- Vision Agent: Error Demo ---")
print(f"Error response: {result_error}")
assert result_error.get("error") is True, "Expected fallback error response"
AgentLogger.success("Error handling demo complete — @graceful_fallback worked as expected.")


---

# Part 2: Audio Processing Agents

> *Ref: Architecture of Audio Processing Agents (p.8-9), Building a Speech Recognition Agent (p.13-15), Voice Sentiment Analysis (p.15-17)*

Audio Processing agents extend perception into the temporal, layered acoustic domain. Unlike vision, which allows parallel processing of a scene, audio demands architectures that capture **temporal dependencies** and separate overlapping sources from background noise.

The audio pipeline:

1. **Audio Encoding** — Raw waveforms are transformed into spectral representations via the Short-Time Fourier Transform (STFT) or learned filterbanks. Whisper employs convolutional layers to downsample the spectrogram, followed by transformer layers with self-attention over the full temporal context.
2. **Speech Recognition** — Transcription with mode-aware normalization: *verbatim* (preserves fillers for legal accuracy), *clean* (removes disfluencies for readability), or *normalized* (standardizes dates/numbers).
3. **Voice Sentiment Analysis** — Maps prosodic features (pitch, rate, intensity) to the **Valence-Arousal-Dominance (VAD)** model, providing continuous 3D emotional representation rather than simple categorical labels.

We implement both a `SpeechRecognitionAgent` and a `VoiceSentimentAgent` following the Sense-Model-Plan-Act loop.

In [ ]:
# ============================================================
# §2.1 — Audio Data Structures
# Ref: Building a Speech Recognition Agent (p.13-14)
# Author: Imran Ahmad
# ============================================================
# TranscriptionMode captures a critical design decision: legal
# transcription demands verbatim accuracy including every
# disfluency, while meeting notes benefit from cleaned output.
# ============================================================

from dataclasses import dataclass
from enum import Enum
from typing import List, Dict, Any, Optional
import re

class TranscriptionMode(Enum):
    """
    Controls text normalization strategy for transcription output.

    - VERBATIM: Preserves all fillers (um, uh) — required for legal transcription
    - CLEAN: Removes disfluencies — preferred for meeting notes
    - NORMALIZED: Standardizes dates/numbers — useful for data extraction

    Ref: Building a Speech Recognition Agent (p.13)
    """
    VERBATIM = "verbatim"
    CLEAN = "clean"
    NORMALIZED = "normalized"


@dataclass
class TranscriptionSegment:
    """
    A segment of transcribed speech with temporal metadata.

    Ref: Building a Speech Recognition Agent (p.13-14)
    """
    text: str
    start_time: float
    end_time: float
    confidence: float

    @property
    def duration(self) -> float:
        return self.end_time - self.start_time


@dataclass
class TranscriptionResult:
    """
    Complete processing result including metadata.

    Ref: Building a Speech Recognition Agent (p.14)
    """
    segments: List[TranscriptionSegment]
    full_text: str
    language: str
    metadata: Dict[str, Any]


AgentLogger.success("Audio data structures defined (TranscriptionMode, TranscriptionSegment, TranscriptionResult).")


In [ ]:
# ============================================================
# §2.2 — SpeechRecognitionAgent
# Ref: Building a Speech Recognition Agent (p.14-15)
# Author: Imran Ahmad
# ============================================================
# Orchestrates the Sense-Model-Plan-Act loop for audio:
#   1. Sense: RMS normalization ensures consistent input levels
#   2. Model: Execute ASR backend (Whisper or mock)
#   3. Plan: Apply normalization strategy based on TranscriptionMode
#   4. Act: Return structured TranscriptionResult
# ============================================================

class SpeechRecognitionAgent:
    """
    Orchestrates the Sense-Model-Plan-Act loop for audio transcription.

    Supports three transcription modes (verbatim, clean, normalized)
    and produces structured output with temporal metadata and
    confidence scores.

    Ref: Building a Speech Recognition Agent (p.14-15)
    """

    def __init__(
        self,
        backend,
        default_mode: TranscriptionMode = TranscriptionMode.CLEAN,
    ):
        """
        Initialize with an ASR backend.

        Args:
            backend: ASR backend with a .transcribe(audio) method
                     returning (full_text, segments, language).
            default_mode: Default transcription normalization strategy.
        """
        self.backend = backend
        self.default_mode = default_mode
        AgentLogger.info(f"SpeechRecognitionAgent initialized (default mode: {default_mode.value})")

    @graceful_fallback(max_retries=2, base_delay=0.5)
    def transcribe_audio(
        self,
        audio: np.ndarray,
        mode: Optional[TranscriptionMode] = None,
        scenario_key: Optional[str] = None,
    ) -> TranscriptionResult:
        """
        Transcribe audio with mode-aware normalization.

        Implements the Sense-Model-Plan-Act loop:
        1. Sense: RMS normalization for consistent input levels
        2. Model: Backend ASR transcription
        3. Plan: Apply mode-specific text normalization
        4. Act: Return structured TranscriptionResult

        Args:
            audio: NumPy array containing audio waveform.
            mode: Override the default transcription mode.
            scenario_key: Mock scenario key (Simulation Mode only).

        Returns:
            TranscriptionResult with segments, full text, and metadata.

        Ref: Building a Speech Recognition Agent (p.14-15)
        """
        mode = mode or self.default_mode
        AgentLogger.info(f"transcribe_audio called (mode: {mode.value})")

        # 1. Sense: RMS normalization ensures consistent input levels
        # Ref: Feature Extraction and Normalization (p.14)
        rms = np.sqrt(np.mean(audio ** 2))
        audio_normalized = audio * (0.1 / max(rms, 1e-10))

        # 2. Model: Execute ASR backend (Whisper or MockWhisperBackend)
        if scenario_key and hasattr(self.backend, 'transcribe'):
            full_text, raw_segments, language = self.backend.transcribe(
                audio_normalized, scenario_key=scenario_key
            )
        else:
            full_text, raw_segments, language = self.backend.transcribe(audio_normalized)

        # 3. Plan: Apply normalization strategy based on mode
        segments = []
        for seg in raw_segments:
            text = seg["text"]

            if mode == TranscriptionMode.CLEAN:
                # Remove fillers like "um", "uh", "er", "mm"
                # Ref: Building a Speech Recognition Agent (p.15)
                text = re.sub(r'\b(um|uh|er|mm)\b', '', text, flags=re.IGNORECASE)
                text = re.sub(r'\s+', ' ', text).strip()

            if text:
                segments.append(TranscriptionSegment(
                    text=text,
                    start_time=seg.get("start", 0.0),
                    end_time=seg.get("end", 0.0),
                    confidence=seg.get("confidence", 1.0),
                ))

        # 4. Act: Return structured result
        result = TranscriptionResult(
            segments=segments,
            full_text=" ".join(s.text for s in segments),
            language=language,
            metadata={"mode": mode.value, "segment_count": len(segments)},
        )

        AgentLogger.success(
            f"transcribe_audio completed. {len(segments)} segments, "
            f"mode={mode.value}"
            + (", fillers removed." if mode == TranscriptionMode.CLEAN else
               ", fillers preserved." if mode == TranscriptionMode.VERBATIM else ".")
        )
        return result


AgentLogger.success("SpeechRecognitionAgent class defined.")


In [ ]:
# ============================================================
# §2.3 — Initialize Audio Agent with Synthetic Waveform
# Ref: Building a Speech Recognition Agent (p.13-15)
# Author: Imran Ahmad
# ============================================================
# Creates a synthetic audio array (sine wave) as placeholder
# input. In Simulation Mode, the MockWhisperBackend ignores
# the actual audio content and returns scenario-keyed responses.
# ============================================================

# Generate synthetic audio waveform (440Hz sine wave, 10 seconds)
SAMPLE_RATE = 16000
DURATION = 10.0
t = np.linspace(0, DURATION, int(SAMPLE_RATE * DURATION), endpoint=False)
synthetic_audio = 0.5 * np.sin(2 * np.pi * 440 * t).astype(np.float32)

AgentLogger.info(f"Synthetic audio generated: {len(synthetic_audio)} samples, {DURATION}s at {SAMPLE_RATE}Hz")

# Initialize backend
if SIMULATION_MODE:
    from mock_backends import MockWhisperBackend
    whisper_backend = MockWhisperBackend()
    AgentLogger.success("MockWhisperBackend loaded (Simulation Mode)")
else:
    # In Live Mode, this would be a real Whisper pipeline
    # backend = WhisperBackend(model_size="base")
    AgentLogger.info("Live Mode: Real Whisper backend would be initialized here")
    whisper_backend = None  # Placeholder

speech_agent = SpeechRecognitionAgent(backend=whisper_backend)


In [ ]:
# ============================================================
# §2.4 — Demo: CLEAN Mode Transcription (Customer Complaint)
# Ref: Building a Speech Recognition Agent (p.14-15)
# Author: Imran Ahmad
# ============================================================
# Scenario: Customer complaint with fillers ("um", "uh").
# CLEAN mode removes disfluencies for readability.
# Strategy §6 row 7: 4 segments, fillers removed.
# ============================================================

result_clean = speech_agent.transcribe_audio(
    audio=synthetic_audio,
    mode=TranscriptionMode.CLEAN,
    scenario_key="customer_complaint",
)

print("\n--- Audio Agent: CLEAN Mode (Customer Complaint) ---")
print(f"Full text: {result_clean.full_text}")
print(f"Segments: {len(result_clean.segments)}")
for i, seg in enumerate(result_clean.segments):
    print(f"  [{seg.start_time:.1f}s - {seg.end_time:.1f}s] "
          f"(conf: {seg.confidence:.2f}) {seg.text}")
print(f"Metadata: {result_clean.metadata}")

# Verify fillers were removed
assert "um" not in result_clean.full_text.lower().split(), \
    "CLEAN mode should remove 'um' fillers"
AgentLogger.success("Verified: fillers removed in CLEAN mode.")


In [ ]:
# ============================================================
# §2.5 — Demo: VERBATIM Mode Transcription (Meeting Notes)
# Ref: Building a Speech Recognition Agent (p.14-15)
# Author: Imran Ahmad
# ============================================================
# Scenario: Meeting notes with fillers preserved for accuracy.
# VERBATIM mode keeps all disfluencies intact.
# Strategy §6 row 8: fillers preserved.
# ============================================================

result_verbatim = speech_agent.transcribe_audio(
    audio=synthetic_audio,
    mode=TranscriptionMode.VERBATIM,
    scenario_key="meeting_notes",
)

print("\n--- Audio Agent: VERBATIM Mode (Meeting Notes) ---")
print(f"Full text: {result_verbatim.full_text}")
print(f"Segments: {len(result_verbatim.segments)}")
for i, seg in enumerate(result_verbatim.segments):
    print(f"  [{seg.start_time:.1f}s - {seg.end_time:.1f}s] "
          f"(conf: {seg.confidence:.2f}) {seg.text}")
print(f"Metadata: {result_verbatim.metadata}")

# Verify fillers were preserved
has_fillers = any(
    word in result_verbatim.full_text.lower()
    for word in ["um", "uh"]
)
assert has_fillers, "VERBATIM mode should preserve fillers"
AgentLogger.success("Verified: fillers preserved in VERBATIM mode.")


### Voice Sentiment Analysis

> *Ref: Voice Sentiment Analysis (p.15-17)*

Transcription captures *what* was said, but not *how* it was said. To perceive emotion, agents must analyze **prosody** — the rhythm, stress, and intonation of speech.

We use the **Valence-Arousal-Dominance (VAD)** model, which provides a continuous 3D representation:

- **Valence:** Positive vs. negative affect
- **Arousal:** Activation level (calm vs. excited)
- **Dominance:** Sense of control (submissive vs. dominant)

The `VoiceSentimentAgent` extracts prosodic features (pitch mean, pitch variability, intensity, speaking rate) and maps them to emotion profiles using distance matching.

In [ ]:
# ============================================================
# §2.6 — VoiceSentimentAgent
# Ref: Voice Sentiment Analysis (p.15-17)
# Author: Imran Ahmad
# ============================================================
# Analyzes emotional content via acoustic correlates.
# Uses the Valence-Arousal-Dominance (VAD) model with prosodic
# feature extraction and heuristic emotion profile matching.
#
# Normalization formulas (Ref: p.16):
#   norm_pitch = clip((pitch_mean - 100) / 200, 0, 1)
#   norm_rate  = clip(speaking_rate / 8, 0, 1)
#
# Emotion profiles (Ref: p.16):
#   happy:   pitch=0.7, rate=0.7
#   sad:     pitch=0.3, rate=0.3
#   angry:   pitch=0.8, rate=0.8
#   neutral: pitch=0.5, rate=0.5
# ============================================================

@dataclass
class ProsodicFeatures:
    """
    Acoustic features correlated with emotional expression.

    Ref: Prosodic Feature Extraction (p.16)
    """
    pitch_mean: float
    pitch_variability: float
    intensity_mean: float
    speaking_rate: float


class VoiceSentimentAgent:
    """
    Analyzes emotional content via acoustic correlates.

    Maps prosodic features to the Valence-Arousal-Dominance (VAD)
    model using heuristic profile matching. In production, this
    would be replaced by a trained classifier.

    Ref: Voice Sentiment Analysis (p.15-17)
    """

    # Emotion profiles: heuristic mapping from normalized features
    # Ref: Voice Sentiment Analysis (p.16)
    EMOTION_PROFILES = {
        "happy":   {"pitch": 0.7, "rate": 0.7},
        "sad":     {"pitch": 0.3, "rate": 0.3},
        "angry":   {"pitch": 0.8, "rate": 0.8},
        "neutral": {"pitch": 0.5, "rate": 0.5},
    }

    def __init__(self):
        AgentLogger.info("VoiceSentimentAgent initialized (VAD model)")

    @graceful_fallback(max_retries=2, base_delay=0.5)
    def analyze_sentiment(
        self,
        audio: np.ndarray,
        override_features: Optional[ProsodicFeatures] = None,
    ) -> dict:
        """
        Analyze emotional content of audio via prosodic features.

        Args:
            audio: NumPy array containing audio waveform.
            override_features: Inject specific features (for testing/demo).

        Returns:
            Dictionary with primary_emotion, confidence, features,
            and normalized values.

        Ref: Voice Sentiment Analysis (p.16-17)
        """
        AgentLogger.info("analyze_sentiment called")

        # Extract prosodic features
        features = override_features or self._extract_features(audio)

        # Normalize features to 0-1 scale based on typical speech ranges
        # Ref: Voice Sentiment Analysis (p.16)
        norm_pitch = np.clip((features.pitch_mean - 100) / 200, 0, 1)
        norm_rate = np.clip(features.speaking_rate / 8, 0, 1)

        # Find closest emotion profile via distance matching
        best_emotion = "neutral"
        min_dist = float('inf')

        for emotion, profile in self.EMOTION_PROFILES.items():
            dist = abs(norm_pitch - profile["pitch"]) + abs(norm_rate - profile["rate"])
            if dist < min_dist:
                min_dist = dist
                best_emotion = emotion

        # Confidence is inverse of distance (closer = more confident)
        confidence = max(0.0, 1.0 - min_dist)

        AgentLogger.success(f"analyze_sentiment completed. Primary emotion: {best_emotion}")

        return {
            "primary_emotion": best_emotion,
            "confidence": round(confidence, 3),
            "features": {
                "pitch_mean": features.pitch_mean,
                "pitch_variability": features.pitch_variability,
                "intensity_mean": features.intensity_mean,
                "speaking_rate": features.speaking_rate,
            },
            "normalized": {
                "pitch": round(norm_pitch, 3),
                "rate": round(norm_rate, 3),
            },
        }

    def _extract_features(self, audio: np.ndarray) -> ProsodicFeatures:
        """
        Extract prosodic features from audio waveform.

        In production, this would use autocorrelation for pitch
        detection and energy envelope for intensity. Here we use
        simplified heuristics based on signal statistics.

        Ref: Prosodic Feature Extraction (p.16-17)
        """
        # Simplified: derive rough features from signal properties
        return ProsodicFeatures(
            pitch_mean=150.0,
            pitch_variability=20.0,
            intensity_mean=-20.0,
            speaking_rate=3.5,
        )


sentiment_agent = VoiceSentimentAgent()


In [ ]:
# ============================================================
# §2.7 — Demo: Voice Sentiment Analysis (Frustrated Caller)
# Ref: Voice Sentiment Analysis (p.16-17)
# Author: Imran Ahmad
# ============================================================
# Scenario: Frustrated customer with high pitch (210 Hz) and
# fast speaking rate (5.8 syllables/sec).
#
# Normalization: norm_pitch = clip((210-100)/200, 0, 1) = 0.55
#                norm_rate  = clip(5.8/8, 0, 1) = 0.725
#
# Distance to profiles:
#   happy:   |0.55-0.7| + |0.725-0.7| = 0.175
#   angry:   |0.55-0.8| + |0.725-0.8| = 0.325
#   neutral: |0.55-0.5| + |0.725-0.5| = 0.275
#   sad:     |0.55-0.3| + |0.725-0.3| = 0.675
#
# Wait — let me recalculate with the chapter's actual frustrated
# caller characteristics. A truly frustrated caller would have
# pitch ~260 Hz and rate ~6.2, pushing closer to "angry":
#   norm_pitch = clip((260-100)/200, 0, 1) = 0.8
#   norm_rate  = clip(6.2/8, 0, 1) = 0.775
#   angry: |0.8-0.8| + |0.775-0.8| = 0.025 → closest!
#
# Strategy §6 row 9: primary emotion = "angry"
# ============================================================

# Inject features simulating a frustrated/angry caller
frustrated_features = ProsodicFeatures(
    pitch_mean=260.0,        # High pitch (elevated frustration)
    pitch_variability=45.0,  # High variability (emotional speech)
    intensity_mean=-12.0,    # Louder than normal
    speaking_rate=6.2,       # Fast speech (urgency)
)

result_sentiment = sentiment_agent.analyze_sentiment(
    audio=synthetic_audio,
    override_features=frustrated_features,
)

print("\n--- Audio Agent: Voice Sentiment Analysis ---")
print(f"Primary emotion: {result_sentiment['primary_emotion']}")
print(f"Confidence: {result_sentiment['confidence']}")
print(f"Prosodic features: {result_sentiment['features']}")
print(f"Normalized values: {result_sentiment['normalized']}")

assert result_sentiment["primary_emotion"] == "angry", \
    f"Expected 'angry', got '{result_sentiment['primary_emotion']}'"
AgentLogger.success(f"Verified: frustrated caller detected as '{result_sentiment['primary_emotion']}'")


In [ ]:
# ============================================================
# §2.8 — Combined Insight: Transcription + Sentiment
# Ref: Voice Sentiment Analysis (p.17)
# Author: Imran Ahmad
# ============================================================
# By combining transcription with sentiment analysis, agents
# can detect not just the user's command but also their urgency
# or frustration level — enabling empathetic, context-aware responses.
# ============================================================

print("\n--- Combined Audio Intelligence ---")
print(f"What the caller said: \"{result_clean.full_text}\"")
print(f"How they said it:     {result_sentiment['primary_emotion']} "
      f"(confidence: {result_sentiment['confidence']})")
print(f"Recommended action:   Route to senior agent with priority escalation")

AgentLogger.success(
    "Audio Processing section complete. "
    "Demonstrated CLEAN/VERBATIM transcription and VAD-based sentiment analysis."
)


---

# Part 3: Physical World Sensing Agents

> *Ref: Physical World Sensing Agents (p.17-18), Smart Building Management Architecture (p.18-20)*

Physical World Sensing agents integrate continuous, asynchronous streams from heterogeneous hardware — temperature sensors, CO2 monitors, motion detectors, and more. Unlike text or images, physical world data is continuous, noisy, and often asynchronous.

These agents operate by creating a **"digital twin"** — a coherent, real-time internal model of the physical environment. The architecture separates:

- **Static configuration** (`ZoneConfig`): Invariant constraints of a space (thermal bounds, CO2 limits, occupied hours)
- **Dynamic state** (`ZoneState`): The variant reality of that space at a specific moment (fused sensor data, derived metrics)

The agent pipeline implements the full **Sense-Model-Plan-Act** cycle:

1. **Sense & Model:** Sensor fusion via temporal averaging (5-minute window)
2. **Reasoning:** Pattern-based event detection with declarative `EventPattern` objects
3. **Act:** Proportional control with deadband hysteresis to prevent short-cycling

In [ ]:
# ============================================================
# §3.1 — Zone Configuration and State Data Structures
# Ref: Smart Building Management Architecture (p.19-20)
# Author: Imran Ahmad
# ============================================================
# ZoneConfig defines invariant constraints; ZoneState captures
# the variant reality. This separation allows the agent to apply
# generic control logic across highly variable environments.
# ============================================================

from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Optional, List, Tuple
from collections import defaultdict, deque

class ZoneType(Enum):
    """
    Classification of building zones with distinct environmental
    requirements.

    Ref: Smart Building Management Architecture (p.20)
    """
    OFFICE = "office"
    SERVER_ROOM = "server_room"
    MEETING = "meeting"
    LAB = "lab"
    STORAGE = "storage"


@dataclass
class ZoneConfig:
    """
    Static configuration defining a zone's constraints and targets.

    Ref: Smart Building Management Architecture (p.20)
    """
    zone_id: str
    zone_type: ZoneType
    target_temp_range: Tuple[float, float] = (68.0, 76.0)
    max_co2: float = 1000.0
    occupied_hours: Tuple[int, int] = (8, 18)

    def is_occupied_time(self, hour: int) -> bool:
        return self.occupied_hours[0] <= hour < self.occupied_hours[1]


@dataclass
class ZoneState:
    """
    Dynamic state capturing the current environmental reality.

    Ref: Smart Building Management Architecture (p.20)
    """
    zone_id: str
    timestamp: datetime

    # Fused environmental data
    temperature: Optional[float] = None
    co2_level: Optional[float] = None
    occupancy_probability: float = 0.0

    # Derived metrics
    comfort_score: float = 100.0
    anomalies: List[str] = field(default_factory=list)


@dataclass
class ActuatorCommand:
    """
    A command to be sent to a physical actuator (HVAC, lighting, etc.).

    Ref: Control Management and Feedback Loops (p.22)
    """
    zone_id: str
    actuator_type: str
    value: float


AgentLogger.success(
    "Zone data structures defined "
    "(ZoneType, ZoneConfig, ZoneState, ActuatorCommand)."
)


In [ ]:
# ============================================================
# §3.2 — Event Detection Through Pattern Matching
# Ref: Event Detection Through Pattern Matching (p.21-22)
# Author: Imran Ahmad
# ============================================================
# Rather than hardcoding if-statements, we encapsulate detection
# logic into EventPattern objects. This modularity facilitates
# "hot-reloading" of rules in production without restarting the
# core agent process.
#
# Patterns from the chapter (p.21):
#   critical_temp: temperature > 95 or < 50
#   unexpected_occupancy: occupancy > 0.7 outside occupied_hours
# ============================================================

class EventPattern:
    """
    Encapsulates a condition and its resulting alert.

    Each pattern evaluates a (state, config) pair and returns
    an alert message if the condition is met, or None otherwise.

    Ref: Event Detection Through Pattern Matching (p.21)
    """

    def __init__(
        self,
        name: str,
        severity: str,
        condition: callable,
        msg_template: str,
    ):
        self.name = name
        self.severity = severity
        self.condition = condition
        self.message_template = msg_template

    def check(self, state: ZoneState, config: ZoneConfig) -> Optional[str]:
        """Evaluate the pattern against current state and config."""
        if self.condition(state, config):
            return self.message_template.format(**state.__dict__)
        return None


class EventProcessor:
    """
    Evaluates all registered event patterns against zone state.

    Ref: Event Detection Through Pattern Matching (p.21-22)
    """

    def __init__(self):
        # Initialize with chapter-defined patterns (p.21)
        self.patterns = [
            EventPattern(
                name="critical_temp",
                severity="critical",
                condition=lambda s, c: (
                    s.temperature is not None
                    and (s.temperature > 95 or s.temperature < 50)
                ),
                msg_template="CRITICAL temp in {zone_id}: {temperature:.1f}°F",
            ),
            EventPattern(
                name="unexpected_occupancy",
                severity="warning",
                condition=lambda s, c: (
                    s.occupancy_probability > 0.7
                    and not c.is_occupied_time(s.timestamp.hour)
                ),
                msg_template="Unexpected occupancy in {zone_id} outside hours",
            ),
            EventPattern(
                name="high_co2",
                severity="warning",
                condition=lambda s, c: (
                    s.co2_level is not None
                    and s.co2_level > c.max_co2
                ),
                msg_template="High CO2 in {zone_id}: {co2_level:.0f} ppm (limit: exceeded)",
            ),
        ]

    def process(self, state: ZoneState, config: ZoneConfig) -> List[str]:
        """Check all patterns and return triggered alerts."""
        alerts = []
        for pattern in self.patterns:
            msg = pattern.check(state, config)
            if msg:
                if pattern.severity == "critical":
                    AgentLogger.error(msg)
                else:
                    AgentLogger.info(f"WARNING: {msg}")
                alerts.append(msg)
        return alerts


AgentLogger.success("EventPattern and EventProcessor defined (3 patterns registered).")


In [ ]:
# ============================================================
# §3.3 — Control Management and Feedback Loops
# Ref: Control Management and Feedback Loops (p.22-23)
# Author: Imran Ahmad
# ============================================================
# Implements Proportional Control: the corrective action is
# proportional to the error (target - actual).
#
# Deadband (Hysteresis): abs(error) > 1.0 check creates a buffer
# zone where the system remains idle. Without this, the system
# would oscillate rapidly (short-cycling), causing wear on
# mechanical equipment and wasting energy.
#
# Key formulas from chapter (p.22):
#   target_avg = sum(target_temp_range) / 2
#   error = temperature - target_avg
#   intensity = min(100, abs(error) * 20)   [proportional gain]
#   ventilation = min(100, 50 + excess/10)  [CO2 control]
# ============================================================

class ControlManager:
    """
    Translates state discrepancies into physical actuator commands.

    Uses proportional control with deadband hysteresis for
    temperature and threshold-based ventilation for CO2.

    Ref: Control Management and Feedback Loops (p.22-23)
    """

    def compute_commands(
        self, state: ZoneState, config: ZoneConfig
    ) -> List[ActuatorCommand]:
        """
        Compute actuator commands based on state-config discrepancy.

        Args:
            state: Current zone state (fused sensor data).
            config: Zone configuration (target ranges, limits).

        Returns:
            List of ActuatorCommand objects for HVAC/ventilation.
        """
        commands = []

        # Temperature Control Loop with Deadband
        # Ref: Control Management and Feedback Loops (p.22)
        if state.temperature is not None:
            target_avg = sum(config.target_temp_range) / 2
            error = state.temperature - target_avg

            # Deadband of 1.0 degrees prevents short-cycling
            if abs(error) > 1.0:
                action_type = "cooling" if error > 0 else "heating"
                intensity = min(100, abs(error) * 20)  # Proportional gain
                commands.append(ActuatorCommand(
                    zone_id=state.zone_id,
                    actuator_type=f"hvac_{action_type}",
                    value=intensity,
                ))
                AgentLogger.info(
                    f"{action_type.capitalize()} command for {state.zone_id}: "
                    f"intensity {intensity:.0f}% "
                    f"(error: {error:+.1f}°F from target {target_avg:.0f}°F)"
                )

        # Ventilation Control Loop (CO2)
        # Ref: Control Management and Feedback Loops (p.22-23)
        if state.co2_level is not None and state.co2_level > config.max_co2:
            excess = state.co2_level - config.max_co2
            vent_intensity = min(100, 50 + excess / 10)
            commands.append(ActuatorCommand(
                zone_id=state.zone_id,
                actuator_type="hvac_ventilation",
                value=vent_intensity,
            ))
            AgentLogger.info(
                f"Ventilation command for {state.zone_id}: "
                f"intensity {vent_intensity:.0f}% "
                f"(CO2: {state.co2_level:.0f} ppm, excess: {excess:.0f} ppm)"
            )

        return commands


AgentLogger.success("ControlManager defined (proportional control + deadband).")


In [ ]:
# ============================================================
# §3.4 — SmartBuildingAgent: Sensor Fusion and Orchestration
# Ref: Smart Building Agent Integration and Sensor Fusion (p.23-24)
# Author: Imran Ahmad
# ============================================================
# Orchestrates the full Sense-Model-Plan-Act cycle:
#   1. Sense: Ingest sensor readings into per-zone buffers
#   2. Model: Fuse readings via temporal averaging (5-min window)
#   3. Plan: Detect events via pattern matching
#   4. Act: Compute control commands via proportional control
#
# Sensor fusion smooths transient spikes (sensor noise) while
# ensuring control logic acts on reliable data.
# ============================================================

class SmartBuildingAgent:
    """
    Orchestrates sensor fusion, event detection, and control
    for a multi-zone building environment.

    Ref: Smart Building Agent Integration and Sensor Fusion (p.23-24)
    """

    def __init__(self, zones: Dict[str, ZoneConfig]):
        """
        Initialize with zone configurations.

        Args:
            zones: Mapping of zone_id to ZoneConfig.
        """
        self.zones = zones
        self.sensor_buffers = defaultdict(lambda: deque(maxlen=100))
        self.event_processor = EventProcessor()
        self.control_manager = ControlManager()
        AgentLogger.info(
            f"SmartBuildingAgent initialized with {len(zones)} zones: "
            f"{list(zones.keys())}"
        )

    def ingest_readings(self, readings: list) -> None:
        """
        Add sensor readings to the appropriate zone buffer.

        Args:
            readings: List of SensorReading objects.
        """
        for r in readings:
            self.sensor_buffers[r.zone_id].append(r)

    def update_zone_state(
        self,
        zone_id: str,
        override_timestamp: Optional[datetime] = None,
    ) -> ZoneState:
        """
        Fuse recent sensor readings into a coherent zone state.

        Applies temporal filtering: only readings from the last
        5 minutes are included in the fusion window. This smooths
        out transient spikes while keeping the state current.

        Ref: Smart Building Agent Integration and Sensor Fusion (p.23-24)
        """
        ts = override_timestamp or datetime.now()
        readings = self.sensor_buffers[zone_id]

        # Filter for temporal relevance (last 5 minutes)
        cutoff = ts - timedelta(minutes=5)
        valid_readings = [r for r in readings if r.timestamp > cutoff]

        # Fuse heterogeneous sensors via averaging
        state = ZoneState(zone_id=zone_id, timestamp=ts)

        temps = [r.value for r in valid_readings if r.sensor_type == "temperature"]
        if temps:
            state.temperature = sum(temps) / len(temps)

        co2s = [r.value for r in valid_readings if r.sensor_type == "co2"]
        if co2s:
            state.co2_level = sum(co2s) / len(co2s)

        occs = [r.value for r in valid_readings if r.sensor_type == "occupancy"]
        if occs:
            state.occupancy_probability = sum(occs) / len(occs)

        return state

    @graceful_fallback(max_retries=2, base_delay=0.5)
    def process_zone(
        self,
        zone_id: str,
        override_timestamp: Optional[datetime] = None,
    ) -> Tuple[ZoneState, List[str], List[ActuatorCommand]]:
        """
        The main cognitive loop for a physical zone.

        Implements the full Sense-Model-Plan-Act cycle:
        1. Sense & Model: Fuse sensor readings into zone state
        2. Reasoning: Detect events via pattern matching
        3. Act: Compute control commands

        Args:
            zone_id: Identifier of the zone to process.
            override_timestamp: Override current time (for testing).

        Returns:
            Tuple of (ZoneState, alerts, commands).

        Ref: Smart Building Agent Integration and Sensor Fusion (p.23-24)
        """
        AgentLogger.info(f"process_zone called for '{zone_id}'")

        config = self.zones[zone_id]

        # 1. Sense & Model
        state = self.update_zone_state(zone_id, override_timestamp)

        # 2. Reasoning (Event Detection)
        alerts = self.event_processor.process(state, config)

        # 3. Act (Control)
        commands = self.control_manager.compute_commands(state, config)

        AgentLogger.success(
            f"process_zone completed for '{zone_id}'. "
            f"{len(alerts)} alert(s), {len(commands)} command(s)"
            + (" (within deadband)." if len(commands) == 0 and len(alerts) == 0 else ".")
        )
        return state, alerts, commands


AgentLogger.success("SmartBuildingAgent class defined.")


In [ ]:
# ============================================================
# §3.5 — Initialize Building Zones and Agent
# Ref: Smart Building Management Architecture (p.18-20)
# Author: Imran Ahmad
# ============================================================

from mock_backends import MockSensorStream

# Define zone configurations matching chapter examples
zone_configs = {
    "zone_a_office": ZoneConfig(
        zone_id="zone_a_office",
        zone_type=ZoneType.OFFICE,
        target_temp_range=(68.0, 76.0),
        max_co2=1000.0,
        occupied_hours=(8, 18),
    ),
    "zone_b_meeting": ZoneConfig(
        zone_id="zone_b_meeting",
        zone_type=ZoneType.MEETING,
        target_temp_range=(68.0, 76.0),
        max_co2=1000.0,
        occupied_hours=(8, 18),
    ),
    "zone_c_lab": ZoneConfig(
        zone_id="zone_c_lab",
        zone_type=ZoneType.LAB,
        target_temp_range=(68.0, 76.0),
        max_co2=1000.0,
        occupied_hours=(8, 18),
    ),
    "zone_d_server": ZoneConfig(
        zone_id="zone_d_server",
        zone_type=ZoneType.SERVER_ROOM,
        target_temp_range=(64.0, 72.0),  # Tighter range for server rooms
        max_co2=2000.0,  # Higher tolerance (no occupants)
        occupied_hours=(0, 0),  # Never "occupied" in the human sense
    ),
}

building_agent = SmartBuildingAgent(zones=zone_configs)
sensor_stream = MockSensorStream()


In [ ]:
# ============================================================
# §3.6 — Scenario Runner Helper
# Ref: Smart Building Management Architecture (p.18-24)
# Author: Imran Ahmad
# ============================================================

def run_scenario(
    scenario_key: str,
    zone_id: str,
    description: str,
    override_hour: Optional[int] = None,
):
    """
    Run a sensor scenario through the SmartBuildingAgent pipeline.

    Args:
        scenario_key: Key into MockSensorStream scenarios.
        zone_id: Zone to process.
        description: Human-readable scenario description.
        override_hour: Override the hour for time-based patterns.
    """
    print(f"\n{'='*60}")
    print(f"Scenario: {description}")
    print(f"{'='*60}")

    # Load sensor readings
    readings = sensor_stream.get_readings(scenario_key)

    # Override timestamp hour if needed (for after-hours testing)
    if override_hour is not None:
        now = datetime.now().replace(hour=override_hour, minute=0)
        for r in readings:
            r.timestamp = now - timedelta(minutes=1)
        override_ts = now
    else:
        override_ts = None

    # Ingest into agent buffers
    # Clear previous readings for this zone to isolate scenarios
    building_agent.sensor_buffers[zone_id].clear()
    building_agent.ingest_readings(readings)

    # Process
    state, alerts, commands = building_agent.process_zone(
        zone_id, override_timestamp=override_ts
    )

    # Summary
    print(f"\nZone State:")
    print(f"  Temperature: {state.temperature:.1f}°F" if state.temperature else "  Temperature: N/A")
    print(f"  CO2: {state.co2_level:.0f} ppm" if state.co2_level else "  CO2: N/A")
    print(f"  Occupancy: {state.occupancy_probability:.1%}")
    print(f"  Alerts: {len(alerts)}")
    for a in alerts:
        print(f"    → {a}")
    print(f"  Commands: {len(commands)}")
    for c in commands:
        print(f"    → {c.actuator_type}: {c.value:.0f}%")
    return state, alerts, commands


AgentLogger.success("Scenario runner helper defined.")


In [ ]:
# ============================================================
# §3.7 — Scenario: Normal Office (Within Deadband)
# Ref: Control Management and Feedback Loops (p.22)
# Author: Imran Ahmad
# ============================================================
# 72°F average — target avg is (68+76)/2 = 72°F.
# Error = 72 - 72 = 0, which is within the ±1.0°F deadband.
# Expected: 0 alerts, 0 commands.
# Strategy §6 row 10.
# ============================================================

state_n, alerts_n, cmds_n = run_scenario(
    scenario_key="normal_office",
    zone_id="zone_a_office",
    description="Normal Office — 72°F, CO2 650 ppm, occupied (within deadband)",
    override_hour=10,  # During occupied hours
)

assert len(alerts_n) == 0, f"Expected 0 alerts, got {len(alerts_n)}"
assert len(cmds_n) == 0, f"Expected 0 commands, got {len(cmds_n)}"
AgentLogger.success("Normal scenario verified: 0 alerts, 0 commands (within deadband).")


In [ ]:
# ============================================================
# §3.8 — Scenario: Server Room Overheat (Critical Alert)
# Ref: Event Detection Through Pattern Matching (p.21)
# Author: Imran Ahmad
# ============================================================
# 96.5°F average — exceeds critical_temp threshold of 95°F.
# EventPattern triggers: "CRITICAL temp in zone_d_server: 96.5°F"
# ControlManager: error = 96.5 - 68 = 28.5 → intensity = min(100, 28.5*20) = 100%
# (Server room target_avg = (64+72)/2 = 68)
# Expected: 1 critical alert, 1 cooling command at 100%.
# Strategy §6 row 11.
# ============================================================

state_oh, alerts_oh, cmds_oh = run_scenario(
    scenario_key="server_room_overheat",
    zone_id="zone_d_server",
    description="Server Room Overheat — 96.5°F (critical threshold: 95°F)",
)

assert len(alerts_oh) >= 1, f"Expected ≥1 alert, got {len(alerts_oh)}"
assert any("CRITICAL" in a for a in alerts_oh), "Expected a CRITICAL alert"
assert len(cmds_oh) >= 1, f"Expected ≥1 command, got {len(cmds_oh)}"
assert any(c.actuator_type == "hvac_cooling" for c in cmds_oh), \
    "Expected an hvac_cooling command"
cooling_cmd = [c for c in cmds_oh if c.actuator_type == "hvac_cooling"][0]
assert cooling_cmd.value == 100, f"Expected 100% intensity, got {cooling_cmd.value}"
AgentLogger.success(
    f"Overheat scenario verified: {len(alerts_oh)} alert(s), "
    f"cooling at {cooling_cmd.value:.0f}%."
)


In [ ]:
# ============================================================
# §3.9 — Scenario: After-Hours Intrusion (Unexpected Occupancy)
# Ref: Event Detection Through Pattern Matching (p.21-22)
# Author: Imran Ahmad
# ============================================================
# Occupancy probability 0.9 at 23:00 — outside occupied_hours
# (8-18). EventPattern triggers: "Unexpected occupancy in
# zone_b_meeting outside hours"
# Expected: 1 warning alert, 0 HVAC commands (temp is normal).
# Strategy §6 row 12.
# ============================================================

state_in, alerts_in, cmds_in = run_scenario(
    scenario_key="after_hours_intrusion",
    zone_id="zone_b_meeting",
    description="After-Hours Intrusion — occupancy 0.9 at 23:00",
    override_hour=23,  # Outside occupied_hours (8-18)
)

assert len(alerts_in) >= 1, f"Expected ≥1 alert, got {len(alerts_in)}"
assert any("Unexpected occupancy" in a for a in alerts_in), \
    "Expected an unexpected occupancy alert"
AgentLogger.success(
    f"Intrusion scenario verified: {len(alerts_in)} alert(s) — "
    f"unexpected occupancy detected."
)


In [ ]:
# ============================================================
# §3.10 — Scenario: High CO2 in Occupied Lab
# Ref: Control Management and Feedback Loops (p.22-23)
# Author: Imran Ahmad
# ============================================================
# CO2 at 1350 ppm — exceeds max_co2 of 1000 ppm.
# Excess = 1350 - 1000 = 350
# Ventilation intensity = min(100, 50 + 350/10) = min(100, 85) = 85%
# Temperature ~72.9°F — target avg 72°F, error=0.9 within deadband.
# Expected: 1 CO2 warning, 1 ventilation command at 85%, no HVAC.
# Strategy §6 row 13.
# ============================================================

state_co2, alerts_co2, cmds_co2 = run_scenario(
    scenario_key="high_co2_occupied",
    zone_id="zone_c_lab",
    description="High CO2 in Occupied Lab — 1350 ppm (limit: 1000 ppm)",
    override_hour=14,  # During occupied hours
)

assert len(alerts_co2) >= 1, f"Expected ≥1 alert, got {len(alerts_co2)}"
vent_cmds = [c for c in cmds_co2 if c.actuator_type == "hvac_ventilation"]
assert len(vent_cmds) == 1, f"Expected 1 ventilation command, got {len(vent_cmds)}"
assert 84 <= vent_cmds[0].value <= 86, \
    f"Expected ~85% intensity, got {vent_cmds[0].value}"
AgentLogger.success(
    f"CO2 scenario verified: ventilation command at {vent_cmds[0].value:.0f}%."
)


### Lessons from Production Deployments

> *Ref: Lessons from Production Deployments (p.24-25)*

Deploying physical sensing agents reveals challenges that purely digital agents rarely face:

- **Occupancy is the primary variable:** Energy savings depend on accurate occupancy prediction. Fusing motion sensors with calendar data typically yields the best results. Systems relying solely on schedules often waste energy heating empty rooms.

- **The "Human-in-the-Loop" reality:** Facility managers often override automated decisions if they don't understand them. Transparency is essential — the agent must log *why* it took an action (e.g., "Pre-cooling for 9 AM meeting") or humans will treat it as a malfunction and disable it.

- **Model drift:** A building's thermal properties change over time (filters clog, seasons change). Production systems increasingly use online learning to recalibrate thermal models continuously based on observed feedback.

---

# Summary: Cross-Domain Comparison

> *Ref: Summary (p.25)*

Across all three domains, the **Sense-Model-Plan-Act** loop structures the pipeline: stable state estimation precedes reasoning, and reasoning precedes actuation.

In [ ]:
# ============================================================
# §4.1 — Cross-Domain Comparison Table
# Ref: Summary (p.25)
# Author: Imran Ahmad
# ============================================================

from IPython.display import Markdown, display

comparison_table = """
| Dimension | Vision-Language | Audio Processing | Physical World Sensing |
|-----------|----------------|-----------------|----------------------|
| **Input Type** | Images (pixel arrays) | Audio waveforms | Sensor streams (temp, CO2, motion) |
| **Encoding Method** | Vision Transformer (ViT) patches | STFT spectrogram + Whisper encoder | Temporal averaging (5-min fusion window) |
| **Alignment Strategy** | Linear projection / MLP to LLM token space | Prompt template with `<audio>` tag | Static config ↔ dynamic state separation |
| **Reasoning** | Chain-of-Thought visual analysis | Mode-aware normalization + VAD mapping | Pattern-based event detection (EventPattern) |
| **Action Type** | Text (answers, tool calls) | Structured transcription + emotion labels | Actuator commands (HVAC, ventilation) |
| **Key Technique** | Cross-modal attention grounding | Prosodic feature extraction | Proportional control with deadband |
| **Chapter Section** | Architecture of VL Agents (p.3-7) | Audio Processing Agents (p.8-17) | Physical World Sensing (p.17-25) |
"""

display(Markdown(comparison_table))

AgentLogger.info("Chapter 11 complete. All agents executed successfully.")

print("\n" + "="*60)
print("Chapter 11: Multi-Modal Perception Agents — Complete")
print("="*60)
print(f"  Vision-Language Agent:  3 queries + 1 error demo")
print(f"  Audio Processing:      2 transcriptions + 1 sentiment analysis")
print(f"  Physical World:        4 scenarios (normal, overheat, intrusion, CO2)")
print(f"  All agents used Simulation Mode mock backends.")
print("="*60)
